In [3]:
"""
Traffic Congestion Prediction Pipeline
Uses YOLOv8 for vehicle detection and LSTM for congestion prediction
"""
import os
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
from ultralytics import YOLO
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ==================== STEP 1: VIDEO PROCESSING WITH YOLO ====================

class TrafficAnalyzer:
    """Extract traffic features from video using YOLO"""
    
    def __init__(self, video_path, model_name='yolov8n.pt'):
        """
        Initialize the traffic analyzer
        
        Args:
            video_path: Path to the traffic video
            model_name: YOLO model to use (yolov8n.pt is fastest)
        """
        self.video_path = video_path
        self.model = YOLO(model_name)
        
        # Vehicle classes in COCO dataset
        self.vehicle_classes = [2, 3, 5, 7]  # car, motorcycle, bus, truck
        self.class_names = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
        
    def extract_features(self, fps_sample=2):
        """
        Extract traffic features from video
        
        Args:
            fps_sample: Process every Nth frame (2 = process at 2 FPS equivalent)
            
        Returns:
            DataFrame with traffic metrics over time
        """
        cap = cv2.VideoCapture(self.video_path)
        
        if not cap.isOpened():
            raise ValueError(f"Cannot open video: {self.video_path}")
        
        # Get video properties
        original_fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_interval = int(original_fps / fps_sample)
        
        print(f"Video FPS: {original_fps}")
        print(f"Total frames: {total_frames}")
        print(f"Processing every {frame_interval} frames (≈{fps_sample} FPS)")
        
        traffic_data = []
        frame_count = 0
        processed_count = 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            # Process every Nth frame
            if frame_count % frame_interval == 0:
                # Run YOLO detection
                results = self.model(frame, verbose=False)
                
                # Extract detections
                detections = results[0].boxes
                
                # Count vehicles by type
                vehicle_counts = defaultdict(int)
                total_vehicles = 0
                bboxes = []
                
                for box in detections:
                    cls = int(box.cls[0])
                    if cls in self.vehicle_classes:
                        vehicle_counts[self.class_names[cls]] += 1
                        total_vehicles += 1
                        
                        # Get bounding box for density calculation
                        bbox = box.xyxy[0].cpu().numpy()
                        bboxes.append(bbox)
                
                # Calculate metrics
                timestamp = frame_count / original_fps
                
                # Calculate average vehicle area (proxy for congestion)
                avg_area = 0
                if bboxes:
                    areas = [(bbox[2]-bbox[0]) * (bbox[3]-bbox[1]) for bbox in bboxes]
                    avg_area = np.mean(areas)
                
                traffic_data.append({
                    'timestamp': timestamp,
                    'frame': frame_count,
                    'total_vehicles': total_vehicles,
                    'cars': vehicle_counts['car'],
                    'motorcycles': vehicle_counts['motorcycle'],
                    'buses': vehicle_counts['bus'],
                    'trucks': vehicle_counts['truck'],
                    'avg_vehicle_area': avg_area
                })
                
                processed_count += 1
                if processed_count % 20 == 0:
                    print(f"Processed {processed_count} frames... ({frame_count}/{total_frames})")
            
            frame_count += 1
        
        cap.release()
        
        df = pd.DataFrame(traffic_data)
        print(f"\nExtracted {len(df)} data points from video")
        return df

# ==================== STEP 2: FEATURE ENGINEERING ====================

class FeatureEngineer:
    """Create features for congestion prediction"""
    
    @staticmethod
    def create_congestion_features(df, window_size=5):
        """
        Create additional features from raw detections
        
        Args:
            df: DataFrame with vehicle counts
            window_size: Window for rolling statistics
            
        Returns:
            DataFrame with engineered features
        """
        df = df.copy()
        
        # Rolling statistics
        df['vehicles_rolling_mean'] = df['total_vehicles'].rolling(window=window_size, min_periods=1).mean()
        df['vehicles_rolling_std'] = df['total_vehicles'].rolling(window=window_size, min_periods=1).std().fillna(0)
        df['vehicles_rolling_max'] = df['total_vehicles'].rolling(window=window_size, min_periods=1).max()
        
        # Rate of change
        df['vehicle_change'] = df['total_vehicles'].diff().fillna(0)
        
        # Congestion score (normalized, 0-1)
        max_vehicles = df['total_vehicles'].max()
        if max_vehicles > 0:
            df['congestion_score'] = df['total_vehicles'] / max_vehicles
        else:
            df['congestion_score'] = 0
        
        # Congestion level (categorical: 0=low, 1=medium, 2=high)
        df['congestion_level'] = pd.cut(df['congestion_score'], 
                                        bins=[0, 0.33, 0.66, 1.0], 
                                        labels=[0, 1, 2], 
                                        include_lowest=True).astype(int)
        
        return df

# ==================== STEP 3: LSTM MODEL ====================

class TrafficDataset(Dataset):
    """PyTorch dataset for time-series traffic data"""
    
    def __init__(self, sequences, targets):
        self.sequences = torch.FloatTensor(sequences)
        self.targets = torch.FloatTensor(targets)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

class CongestionLSTM(nn.Module):
    """LSTM model for congestion prediction"""
    
    def __init__(self, input_size, hidden_size=64, num_layers=2, output_size=1, dropout=0.2):
        super(CongestionLSTM, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, output_size)
        )
    
    def forward(self, x):
        # LSTM layer
        lstm_out, _ = self.lstm(x)
        
        # Take the last time step
        last_output = lstm_out[:, -1, :]
        
        # Fully connected layers
        output = self.fc(last_output)
        
        return output

class CongestionPredictor:
    """Train and predict with LSTM"""
    
    def __init__(self, sequence_length=10, prediction_horizon=5):
        """
        Args:
            sequence_length: Number of past time steps to use
            prediction_horizon: Number of time steps ahead to predict
        """
        self.sequence_length = sequence_length
        self.prediction_horizon = prediction_horizon
        self.scaler = MinMaxScaler()
        self.model = None
        self.feature_columns = None
        
    def prepare_sequences(self, df):
        """Create sequences for LSTM training"""
        
        # Select features for prediction
        self.feature_columns = ['total_vehicles', 'cars', 'motorcycles', 'buses', 
                               'trucks', 'avg_vehicle_area', 'vehicles_rolling_mean',
                               'vehicles_rolling_std', 'vehicle_change']
        
        # Scale features
        scaled_data = self.scaler.fit_transform(df[self.feature_columns])
        
        # Create sequences
        sequences = []
        targets = []
        
        for i in range(len(scaled_data) - self.sequence_length - self.prediction_horizon + 1):
            # Input: past sequence_length time steps
            seq = scaled_data[i:i + self.sequence_length]
            
            # Target: congestion score prediction_horizon steps ahead
            target_idx = i + self.sequence_length + self.prediction_horizon - 1
            target = df.iloc[target_idx]['congestion_score']
            
            sequences.append(seq)
            targets.append(target)
        
        return np.array(sequences), np.array(targets)
    
    def train(self, df, epochs=100, batch_size=32, learning_rate=0.001):
        """Train the LSTM model"""
        
        print("\n" + "="*50)
        print("TRAINING LSTM MODEL")
        print("="*50)
        
        # Prepare data
        sequences, targets = self.prepare_sequences(df)
        
        # Split train/validation
        X_train, X_val, y_train, y_val = train_test_split(
            sequences, targets, test_size=0.2, random_state=42, shuffle=False
        )
        
        print(f"Training samples: {len(X_train)}")
        print(f"Validation samples: {len(X_val)}")
        
        # Create datasets
        train_dataset = TrafficDataset(X_train, y_train)
        val_dataset = TrafficDataset(X_val, y_val)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # Initialize model
        input_size = len(self.feature_columns)
        self.model = CongestionLSTM(input_size=input_size)
        
        # Loss and optimizer
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=learning_rate)
        
        # Training loop
        train_losses = []
        val_losses = []
        
        for epoch in range(epochs):
            # Training
            self.model.train()
            train_loss = 0
            for sequences, targets in train_loader:
                optimizer.zero_grad()
                outputs = self.model(sequences)
                loss = criterion(outputs.squeeze(), targets)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            
            train_loss /= len(train_loader)
            train_losses.append(train_loss)
            
            # Validation
            self.model.eval()
            val_loss = 0
            with torch.no_grad():
                for sequences, targets in val_loader:
                    outputs = self.model(sequences)
                    loss = criterion(outputs.squeeze(), targets)
                    val_loss += loss.item()
            
            val_loss /= len(val_loader)
            val_losses.append(val_loss)
            
            if (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        return train_losses, val_losses
    
    def predict(self, df):
        """Make predictions on new data"""
        
        sequences, actual = self.prepare_sequences(df)
        
        self.model.eval()
        with torch.no_grad():
            sequences_tensor = torch.FloatTensor(sequences)
            predictions = self.model(sequences_tensor).squeeze().numpy()
        
        return predictions, actual

# ==================== STEP 4: VISUALIZATION ====================

class Visualizer:
    """Visualize results"""
    
    @staticmethod
    def plot_traffic_analysis(df, save_path='traffic_analysis.png'):
        """Plot traffic detection results"""
        
        fig, axes = plt.subplots(3, 1, figsize=(14, 10))
        
        # Vehicle counts over time
        axes[0].plot(df['timestamp'], df['total_vehicles'], label='Total Vehicles', linewidth=2)
        axes[0].fill_between(df['timestamp'], df['total_vehicles'], alpha=0.3)
        axes[0].set_xlabel('Time (seconds)')
        axes[0].set_ylabel('Vehicle Count')
        axes[0].set_title('Vehicle Detection Over Time')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Vehicle types breakdown
        axes[1].plot(df['timestamp'], df['cars'], label='Cars', linewidth=1.5)
        axes[1].plot(df['timestamp'], df['motorcycles'], label='Motorcycles', linewidth=1.5)
        axes[1].plot(df['timestamp'], df['buses'], label='Buses', linewidth=1.5)
        axes[1].plot(df['timestamp'], df['trucks'], label='Trucks', linewidth=1.5)
        axes[1].set_xlabel('Time (seconds)')
        axes[1].set_ylabel('Count')
        axes[1].set_title('Vehicle Types Over Time')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        # Congestion score
        axes[2].plot(df['timestamp'], df['congestion_score'], color='red', linewidth=2)
        axes[2].fill_between(df['timestamp'], df['congestion_score'], alpha=0.3, color='red')
        axes[2].axhline(y=0.33, color='green', linestyle='--', label='Low/Medium threshold')
        axes[2].axhline(y=0.66, color='orange', linestyle='--', label='Medium/High threshold')
        axes[2].set_xlabel('Time (seconds)')
        axes[2].set_ylabel('Congestion Score')
        axes[2].set_title('Congestion Level Over Time')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved traffic analysis plot to {save_path}")
        plt.close()
    
    @staticmethod
    def plot_predictions(actual, predicted, save_path='predictions.png'):
        """Plot LSTM predictions vs actual"""
        
        fig, axes = plt.subplots(2, 1, figsize=(14, 8))
        
        # Predictions vs Actual
        axes[0].plot(actual, label='Actual Congestion', linewidth=2, alpha=0.7)
        axes[0].plot(predicted, label='Predicted Congestion', linewidth=2, alpha=0.7)
        axes[0].set_xlabel('Time Step')
        axes[0].set_ylabel('Congestion Score')
        axes[0].set_title('LSTM Predictions vs Actual Congestion')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Error
        error = actual - predicted
        axes[1].plot(error, color='red', linewidth=1.5)
        axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
        axes[1].fill_between(range(len(error)), error, alpha=0.3, color='red')
        axes[1].set_xlabel('Time Step')
        axes[1].set_ylabel('Prediction Error')
        axes[1].set_title('Prediction Error (Actual - Predicted)')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved predictions plot to {save_path}")
        plt.close()
    
    @staticmethod
    def plot_training_loss(train_losses, val_losses, save_path='training_loss.png'):
        """Plot training and validation loss"""
        
        plt.figure(figsize=(10, 6))
        plt.plot(train_losses, label='Training Loss', linewidth=2)
        plt.plot(val_losses, label='Validation Loss', linewidth=2)
        plt.xlabel('Epoch')
        plt.ylabel('Loss (MSE)')
        plt.title('LSTM Training Progress')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved training loss plot to {save_path}")
        plt.close()

# ==================== MAIN PIPELINE ====================

def run_pipeline(video_path, output_dir='./traffic_outputs'):

    # ✅ CREATE OUTPUT DIRECTORY IF IT DOESN'T EXIST
    os.makedirs(output_dir, exist_ok=True)
    """
    Complete pipeline: YOLO detection -> Feature engineering -> LSTM prediction
    
    Args:
        video_path: Path to traffic video
        output_dir: Directory to save outputs
    """
    
    print("\n" + "="*70)
    print("TRAFFIC CONGESTION PREDICTION PIPELINE")
    print("="*70)
    
    # Step 1: Extract features with YOLO
    print("\n[STEP 1] Running YOLO vehicle detection...")
    analyzer = TrafficAnalyzer(video_path, model_name='yolov8n.pt')
    traffic_df = analyzer.extract_features(fps_sample=2)
    
    # Save raw data
    csv_path = f'{output_dir}/traffic_data.csv'
    traffic_df.to_csv(csv_path, index=False)
    print(f"Saved traffic data to {csv_path}")
    
    # Step 2: Feature engineering
    print("\n[STEP 2] Engineering features...")
    engineer = FeatureEngineer()
    traffic_df = engineer.create_congestion_features(traffic_df, window_size=5)
    
    # Display summary statistics
    print("\nTraffic Summary Statistics:")
    print(f"Average vehicles: {traffic_df['total_vehicles'].mean():.2f}")
    print(f"Max vehicles: {traffic_df['total_vehicles'].max()}")
    print(f"Average congestion score: {traffic_df['congestion_score'].mean():.2f}")
    
    congestion_dist = traffic_df['congestion_level'].value_counts().sort_index()
    print("\nCongestion Distribution:")
    print(f"Low (0): {congestion_dist.get(0, 0)} time steps")
    print(f"Medium (1): {congestion_dist.get(1, 0)} time steps")
    print(f"High (2): {congestion_dist.get(2, 0)} time steps")
    
    # Visualize traffic analysis
    Visualizer.plot_traffic_analysis(traffic_df, 
                                     save_path=f'{output_dir}/traffic_analysis.png')
    
    # Step 3: Train LSTM
    print("\n[STEP 3] Training LSTM for congestion prediction...")
    predictor = CongestionPredictor(sequence_length=10, prediction_horizon=5)
    
    # Train model
    train_losses, val_losses = predictor.train(traffic_df, epochs=100, batch_size=16)
    
    # Visualize training
    Visualizer.plot_training_loss(train_losses, val_losses,
                                  save_path=f'{output_dir}/training_loss.png')
    
    # Step 4: Make predictions
    print("\n[STEP 4] Making predictions...")
    predictions, actual = predictor.predict(traffic_df)
    
    # Calculate metrics
    mse = np.mean((actual - predictions) ** 2)
    mae = np.mean(np.abs(actual - predictions))
    rmse = np.sqrt(mse)
    
    print(f"\nPrediction Metrics:")
    print(f"MSE: {mse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    
    # Visualize predictions
    Visualizer.plot_predictions(actual, predictions,
                                save_path=f'{output_dir}/predictions.png')
    
    print("\n" + "="*70)
    print("PIPELINE COMPLETE!")
    print("="*70)
    print(f"\nOutputs saved to: {output_dir}")
    print("- traffic_data.csv: Raw detection data")
    print("- traffic_analysis.png: Vehicle detection visualization")
    print("- training_loss.png: LSTM training progress")
    print("- predictions.png: Congestion predictions")
    
    return traffic_df, predictor

# ==================== USAGE EXAMPLE ====================

if __name__ == "__main__":
    # Replace with your video path
    VIDEO_PATH = "traffic_video.mov"
    
    # Run the complete pipeline
    traffic_data, trained_model = run_pipeline(VIDEO_PATH)
    
    print("\n✅ All done! Check the output files for results.")


TRAFFIC CONGESTION PREDICTION PIPELINE

[STEP 1] Running YOLO vehicle detection...
Video FPS: 29.999528203330886
Total frames: 22255
Processing every 14 frames (≈2 FPS)
Processed 20 frames... (266/22255)
Processed 40 frames... (546/22255)
Processed 60 frames... (826/22255)
Processed 80 frames... (1106/22255)
Processed 100 frames... (1386/22255)
Processed 120 frames... (1666/22255)
Processed 140 frames... (1946/22255)
Processed 160 frames... (2226/22255)
Processed 180 frames... (2506/22255)
Processed 200 frames... (2786/22255)
Processed 220 frames... (3066/22255)
Processed 240 frames... (3346/22255)
Processed 260 frames... (3626/22255)
Processed 280 frames... (3906/22255)
Processed 300 frames... (4186/22255)
Processed 320 frames... (4466/22255)
Processed 340 frames... (4746/22255)
Processed 360 frames... (5026/22255)
Processed 380 frames... (5306/22255)
Processed 400 frames... (5586/22255)
Processed 420 frames... (5866/22255)
Processed 440 frames... (6146/22255)
Processed 460 frames...